# Heart Attack Prediction: Ensemble Learning Approach

## Overview
This project aims to predict the likelihood of heart disease in patients using clinical data. We employ an ensemble voting classifier that combines multiple machine learning models (Random Forest, SVM, Gradient Boosting, XGBoost, and LightGBM) to achieve robust and accurate predictions.

## Key Steps:
1. **Data Preprocessing**: Handling missing values and scaling numerical features.
2. **Feature Engineering**: Creating domain-specific features to capture physiological relationships.
3. **Model Selection**: Using a `VotingClassifier` for an optimized ensemble approach.
4. **Evaluation**: Assessing model performance using F1-score and cross-validation.

## 1. Environment Setup & Data Loading
We start by importing the necessary libraries for data manipulation, preprocessing, and machine learning.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

# Load data
# Note: In a production environment, these paths would be managed more dynamically
try:
    train_df = pd.read_csv("train.csv")
    test_df = pd.read_csv("test_X.csv")
    print("Data loaded successfully.")
except FileNotFoundError:
    print("Data files not found. Please ensure 'train.csv' and 'test_X.csv' are in the project directory.")

## 2. Data Cleaning & Preprocessing
We handle patient identifiers, treat zero values as missing data where appropriate, and perform median imputation.

In [ ]:
# Drop PatientID column as it lacks predictive value
if 'PatientID' in train_df.columns:
    train_df = train_df.drop(columns=["PatientID"])
if 'PatientID' in test_df.columns:
    test_ids = test_df["PatientID"]
    test_df = test_df.drop(columns=["PatientID"])

# Replace physiological 0s with NaN for more accurate imputation
cols_to_clean = ['Cholesterol', 'RestingBP']
for col in cols_to_clean:
    train_df[col] = train_df[col].replace(0, np.nan)
    test_df[col] = test_df[col].replace(0, np.nan)

# Impute missing values using the median strategy
imputer = SimpleImputer(strategy='median')
train_df[cols_to_clean] = imputer.fit_transform(train_df[cols_to_clean])
test_df[cols_to_clean] = imputer.transform(test_df[cols_to_clean])

## 3. Feature Engineering
Creating normalized features to better capture relationships between age, blood pressure, and cholesterol levels.

In [ ]:
def engineer_features(df):
    df['CholesterolPerAge'] = df['Cholesterol'] / df['Age']
    df['RestingBPPerAge'] = df['RestingBP'] / df['Age']
    df['MaxHRPerAge'] = df['MaxHR'] / df['Age']
    df['RestingBPPerCholesterol'] = df['RestingBP'] / df['Cholesterol']
    df['MaxHRPerRestingBP'] = df['MaxHR'] / df['RestingBP']
    df['Oldpeak_significant'] = (df['Oldpeak'] > 0).astype(int)
    return df

train_df = engineer_features(train_df)
test_df = engineer_features(test_df)

## 4. Modeling Pipeline
We define a robust pipeline that includes scaling, encoding, and our ensemble classifier.

In [ ]:
# Feature categorization
categorical_features = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
numerical_features = [
    "Age", "RestingBP", "Cholesterol", "FastingBS", "MaxHR", "Oldpeak",
    "CholesterolPerAge", "MaxHRPerAge", "RestingBPPerCholesterol"
]

# Preprocessing definition
preprocessor = ColumnTransformer([
    ('num', RobustScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# Defining base models with calibrated probabilities
rf = CalibratedClassifierCV(RandomForestClassifier(
    class_weight='balanced', max_depth=10, min_samples_split=5,
    n_estimators=150, random_state=42), method='isotonic', cv=3)

svm = CalibratedClassifierCV(SVC(
    C=2, kernel='rbf', gamma='auto', degree=2,
    class_weight=None, probability=True), method='sigmoid', cv=3)

gb = CalibratedClassifierCV(GradientBoostingClassifier(
    n_estimators=150, max_depth=3, learning_rate=0.05,
    min_samples_split=20, min_samples_leaf=1,
    subsample=0.8, random_state=123), method='sigmoid', cv=3)

xgb = XGBClassifier(
    objective='binary:logistic', eval_metric='logloss',
    class_weight='balanced', max_depth=3, learning_rate=0.05,
    n_estimators=300, subsample=0.8, tree_method='exact', random_state=123)

lgbm = LGBMClassifier(
    learning_rate=0.05, max_depth=5, n_estimators=100,
    num_leaves=31, objective='binary',
    class_weight='balanced', random_state=42)

# Ensemble Voting Classifier
voting_clf = VotingClassifier(
    estimators=[
        ('gb', gb),
        ('rf', rf),
        ('svm', svm),
        ('xgb', xgb),
        ('lgbm', lgbm)
    ],
    voting='soft'
)

# Integration into a final pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', voting_clf)
])

## 5. Training & Evaluation
We fit the model and evaluate its performance using F1-score and cross-validation to ensure generalization.

In [ ]:
X = train_df.drop(columns=["HeartDisease"])
y = train_df["HeartDisease"]

# Model Training
pipeline.fit(X, y)

# Performance Assessment
train_preds = pipeline.predict(X)
train_f1 = f1_score(y, train_preds)
print(f"Final Training F1 Score: {train_f1:.4f}")

# Cross-Validation
cv_scores = cross_val_score(pipeline, X, y, scoring='f1', cv=5)
print(f"Cross-Validation F1 Scores: {cv_scores}")
print(f"Mean CV F1 Score: {np.mean(cv_scores):.4f}")

## 6. Conclusion
The ensemble approach demonstrates strong predictive capability. This model can serve as a supportive tool for clinical decision-making by identifying high-risk patients based on standard medical metrics.